In [1]:

import numpy as np
import glob
import cv2
import re
from tqdm import tqdm
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import pickle
import seaborn as sns
from sklearn.cluster import DBSCAN
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')
from scipy.optimize import minimize
# Increase chunksize
mpl.rcParams['agg.path.chunksize'] = 10000  # Adjust as needed
# Optionally enable path simplification
mpl.rcParams['path.simplify'] = True
mpl.rcParams['path.simplify_threshold'] = 0.1


In [2]:
# Load your pickle file
exp_name = '20241107_1
pickle_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
output_folder_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
# pickle_path = '/Volumes/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
# output_folder_path = '/Volumes/ReiterU/Anouk/basler_data/'+ exp_name+ '/'



#output_folder_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
local_folder_path = "/Users/anoukberaud/Library/CloudStorage/OneDrive-OIST/Stephens_rotation/research_results/" + exp_name + '/'
pickle_name_aruco = exp_name + '_aruco_panorama_frame.pkl'

pickle_name_sleap = exp_name + '_sleap_panorama_frame.pkl'
col_1, col_2 = 9, 11

In [ ]:
  
with open(output_folder_path + pickle_name_aruco, 'rb') as f:
    aruco_df = pickle.load(f)
    

In [ ]:
# with open(pickle_path + pickle_name_sleap, 'rb') as f:
#     sleap_df = pickle.load(f)

with open(output_folder_path + pickle_name_sleap, 'rb') as f:
    sleap_df = pickle.load(f)


In [ ]:

df_1_aruco = aruco_df[aruco_df['Colony_name'] == col_1]
df_2_aruco = aruco_df[aruco_df['Colony_name'] == col_2]



df_1_sleap = sleap_df[sleap_df['Colony_name'] == col_1]
df_2_sleap = sleap_df[sleap_df['Colony_name'] == col_2]


In [ ]:
def show_original_alignment(df, col):
    
    grouped_by_cam = df.groupby('Cam')
    
    plt.figure(figsize=(15, 15))
    
    for cam, data in grouped_by_cam: 
        plt.scatter(data['X'], data['Y'], s = 1, label = cam)
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()  # Adjust layout to avoid clipping
    plt.title('Original aligment for Colony' + str(col))
    plt.show()



In [ ]:
show_original_alignment(df_1_aruco, col_1)
show_original_alignment(df_1_sleap, col_1)
show_original_alignment(df_2_aruco, col_1)
show_original_alignment(df_2_sleap, col_1)

In [ ]:
def align_cams(df, camera_order, col):
        
    def find_overlapping_points(ref_data, target_data):
        """
        Find overlapping points between reference and target cameras based on
        matching ARUCO_number and Frame_number.
        
        Args:
            ref_data (DataFrame): Data for the reference camera.
            target_data (DataFrame): Data for the target camera.
        
        Returns:
            DataFrame, DataFrame: Filtered data for reference and target cameras.
        """
        # Merge on ARUCO_number and Frame_number to find matches
        merged_data = pd.merge(
            ref_data,
            target_data,
            on=['ARUCO_number', 'Frame_number'],
            suffixes=('_ref', '_target')
        )
        
        # Extract matched points
        ref_points = merged_data[['X_ref', 'Y_ref']]
        target_points = merged_data[['X_target', 'Y_target']]
        return ref_points, target_points

    def align_camera_with_overlap(ref_data, target_data):
        """
        Align target_data to ref_data by minimizing distances for overlapping points.
        
        Args:
            ref_data (DataFrame): Reference camera data with X, Y, ARUCO_number, and Frame_number.
            target_data (DataFrame): Target camera data to align.
        
        Returns:
            tuple: Optimal translation (dx, dy).
        """
        # Find overlapping points
        ref_points, target_points = find_overlapping_points(ref_data, target_data)
        ref_points = ref_points.values
        target_points = target_points.values
        
        def objective(translation):
            dx, dy = translation
            transformed_target = target_points + np.array([dx, dy])
            distances = np.linalg.norm(ref_points - transformed_target, axis=1)
            return np.sum(distances)
    
        # Initial guess for translation (dx=0, dy=0)
        initial_translation = [0, 0]
        result = minimize(objective, initial_translation, method='Powell')
        return result.x  # Optimized dx, dy


    # Full alignment process
    realigned_df = df.copy()
    translations = {cam: (0, 0) for cam in camera_order}  # Store translations

    for i in range(1, len(camera_order)):
        ref_cam = camera_order[i - 1]
        target_cam = camera_order[i]
        
        ref_data = realigned_df[realigned_df['Cam'] == ref_cam]
        target_data = realigned_df[realigned_df['Cam'] == target_cam]
        
        dx, dy = align_camera_with_overlap(ref_data, target_data)
        translations[target_cam] = (dx, dy)
        
        realigned_df.loc[realigned_df['Cam'] == target_cam, 'X'] += dx
        realigned_df.loc[realigned_df['Cam'] == target_cam, 'Y'] += dy

    # Plotting the aligned data
    plt.figure(figsize=(15, 15))
    for cam, data in realigned_df.groupby('Cam'):
        plt.scatter(data['X'], data['Y'], s=1, label=cam)
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Camera ID")
    plt.title('New alignement for Colony ' + str(col))
    plt.tight_layout()
    plt.show()

    return realigned_df, translations


In [ ]:
# Define the order of alignment
#camera_order_1 = [0,1,2,7,6,5,10,11,12,17, 16, 15, 20, 21,22]
camera_order_1 = [20,21,22, 17,16,15, 10,11,12, 7,6,5,0,1,2]
#camera_order_2 = [ 22,23,24, 19,18,17, 12, 13,14, 9,8,3,4]  
camera_order_2 = [22,23,24, 19,18,17, 12, 13,14, 9,8,7,2,3,4]
aligned_df_1_aruco, translations_1 = align_cams(df_1_aruco, camera_order_1, col_1)
aligned_df_2_aruco, translations_2 = align_cams(df_2_aruco, camera_order_2, col_2)

In [ ]:
def apply_translations(df, translations, col):
    """
    Apply given translations to a DataFrame to align its camera data.

    Args:
        df (DataFrame): The DataFrame containing the camera data to be aligned.
        translations (dict): Dictionary of translations {camera_id: (dx, dy)}.
        col (str): Column name or identifier for the dataset (for plot title).

    Returns:
        DataFrame: The realigned DataFrame with updated X and Y values.
    """
    # Copy the DataFrame to avoid modifying the original
    aligned_df = df.copy()

    # Apply translations to each camera
    for cam, (dx, dy) in translations.items():
        aligned_df.loc[aligned_df['Cam'] == cam, 'X'] += dx
        aligned_df.loc[aligned_df['Cam'] == cam, 'Y'] += dy

    # Plot the aligned data
    plt.figure(figsize=(15, 15))
    for cam, data in aligned_df.groupby('Cam'):
        plt.scatter(data['X'], data['Y'], s=1, label=cam)
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Camera ID")
    plt.title(f'Aligned Data for {col}')
    plt.tight_layout()
    plt.show()

    return aligned_df


In [ ]:
aligned_df_1_sleap = apply_translations(df_1_sleap, translations_1, col_1)
aligned_df_2_sleap = apply_translations(df_2_sleap, translations_2, col_2)

In [ ]:
def convert_to_pickle_with_dump(folder_path, pickle_name, df_to_convert): 
    """
    Save a DataFrame to a pickle file using `pickle.dump`, ensuring the folder exists.

    Parameters:
    - folder_path (str): Path to the folder where the pickle file will be saved.
    - pickle_name (str): Name of the pickle file (including '.pkl').
    - df_to_convert (pd.DataFrame): DataFrame to save.

    Returns:
    None
    """
    # Ensure the folder exists, create it if not
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder '{folder_path}' created.")
    else:
        print(f"Folder '{folder_path}' already exists.")
    
    # Full path to the pickle file
    pickle_full_path = os.path.join(folder_path, pickle_name)
    
    try:
        # Save the DataFrame to a pickle file using `pickle.dump`
        print(f"Saving DataFrame to {pickle_full_path} using `pickle.dump`...")
        with open(pickle_full_path, 'wb') as f:
            pickle.dump(df_to_convert, f)
        print(f"File '{pickle_name}' saved successfully.")
    except Exception as e:
        print(f"Error while saving file: {e}")

In [ ]:
convert_to_pickle_with_dump(output_folder_path, exp_name + '_sleap_realigned_col1.pkl', aligned_df_1_sleap)

convert_to_pickle_with_dump(output_folder_path, exp_name + '_sleap_realigned_col2.pkl', aligned_df_2_sleap)


In [ ]:
convert_to_pickle_with_dump(output_folder_path, exp_name + '_aruco_realigned_col1.pkl', aligned_df_1_aruco)

convert_to_pickle_with_dump(output_folder_path, exp_name + '_aruco_realigned_col2.pkl', aligned_df_2_aruco)

